In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
repo_root=Path(".")
os.chdir(repo_root)
from scripts import compute_UVLF_cv as cv
from scripts import groupcat

# Jacob's plot style
plt.style.use("classic")
plt.rc("xtick", labelsize=16)
plt.rc("ytick", labelsize=16)
plt.rc("xtick.major", size=14, width=2)
plt.rc("xtick.minor", size=7, width=2, visible=True)
plt.rc("ytick.major", size=14, width=2)
plt.rc("ytick.minor", size=7, width=2, visible=True)
plt.rc("lines", linewidth=2, markersize=10)
plt.rc("axes", linewidth=2, labelsize=18, labelpad=2.5)
plt.rc("legend", fontsize=13, loc="best", frameon=True, numpoints=1, scatterpoints=1)
plt.rc("font", family="STIXGeneral")
plt.rc("mathtext", fontset="stix")

plt.rcParams["figure.facecolor"]="white"
plt.rcParams["axes.facecolor"]="white"
plt.rcParams["savefig.facecolor"]="white"
plt.rcParams["savefig.transparent"]=False

In [ ]:
#load data
lcdm_data=cv.load_data(cv.lcdm_base_path,cv.snap_num)
lcdm_positions_cmpc,lcdm_MUV,lcdm_subhalo_mass,lcdm_valid=cv.select_valid_uv_galaxies(lcdm_data,run_type="LCDM")

In [ ]:
#box and UVLF calculation
lcdm_volume,lcdm_volume_error,lcdm_los_depth=cv.calculate_field_volume(cosmology=cv.lumina_cosmology,z_min=cv.z_min,z_max=cv.z_max,survey_area=cv.survey_area)
lcdm_field_lengths_cmpc=cv.calculate_rectangle_field_lengths(cosmology=cv.lumina_cosmology,z_center=cv.z_center,z_min=cv.z_min,z_max=cv.z_max,width_angle=cv.nircam_width,height_angle=cv.nircam_height)
lcdm_field_volume=np.prod(lcdm_field_lengths_cmpc)*u.Mpc**3

print("integrated field volume:",lcdm_volume)
print("rectangular field volume:",lcdm_field_volume)
print("field dimensions:",lcdm_field_lengths_cmpc)

lcdm_field_ids,lcdm_inside,_,lcdm_n_fields=cv.assign_subhalos_to_fields(positions_cmpc=lcdm_positions_cmpc,box_size_cmpc=lcdm_data["box_size_cmpc"],field_volume=lcdm_field_volume,geometry=cv.geometry,field_lengths_cmpc=lcdm_field_lengths_cmpc)
lcdm_MUV_inside=lcdm_MUV[lcdm_inside]
lcdm_subhalo_mass_inside=lcdm_subhalo_mass[lcdm_inside]

print("number of fields:",lcdm_n_fields)
print("galaxies inside tiled volume:",len(lcdm_MUV_inside))
print("field IDs:",len(lcdm_field_ids))

lcdm_MUV_bin_centers,lcdm_log_phi=cv.compute_UVLF(MUV_values=lcdm_MUV_inside,field_volume=lcdm_field_volume,n_fields=lcdm_n_fields,MUV_bin_edges=cv.MUV_bin_edges,MUV_bin_width=cv.MUV_bin_width)

In [ ]:
fig,ax=plt.subplots(figsize=(9,7))
ax.plot(lcdm_MUV_bin_centers,lcdm_log_phi,marker="s",linestyle="-",color="gray",label="LUMINA LCDM")
cv.obs.plot_photoz_constraints(redshift=10,ax=ax,marker="o",ms=5,linestyle="none",color="black",capsize=4)
cv.obs.plot_specz_constraints(redshift=10,ax=ax,marker="s",ms=5,linestyle="none",color="red",capsize=4)
ax.set_xlabel(r"$M_{\rm UV}$")
ax.set_ylabel(r"$\log_{10}(\Phi/{\rm Mpc}^{-3}\,{\rm mag}^{-1})$")
ax.set_xlim(-24,-16)
ax.set_ylim(-7.5,0.5)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
data=groupcat.loadSubhalos(cv.lcdm_base_path,cv.snap_num,fields=["SubhaloMass","SubhaloSfrInRad","SubhaloGroupNr"],)
halos=groupcat.loadHalos(cv.lcdm_base_path,cv.snap_num,fields=["GroupFirstSub","Group_M_Crit200","Group_M_TopHat200"],)

lcdm_subhalo_mass_msun=data["SubhaloMass"][lcdm_valid]*1e10/lcdm_data["h"] #msun unit
lcdm_sfr=data["SubhaloSfrInRad"][lcdm_valid] #msun/yr unit
# lcdm_gas_mass=data["SubhaloMassInRadType"][lcdm_valid][:,0]*1e10/lcdm_data["h"] #msun
# lcdm_stellar_mass=data["SubhaloMassInRadType"][lcdm_valid][:,4]*1e10/lcdm_data["h"] #msun
# lcdm_stellar_halfmass_radius=data["SubhaloHalfmassRadType"][lcdm_valid][:,4]/(lcdm_data["h"]*(1+lcdm_data["redshift"])) #proper kpc

# lcdm_radius=2*lcdm_stellar_halfmass_radius

In [ ]:
#halo scale instantaneous SFE model (Shen et al 2026)
#finding the central subhalos
subhalo_index=np.arange(len(data["SubhaloGroupNr"]))
central=subhalo_index==halos["GroupFirstSub"][data["SubhaloGroupNr"]]
lcdm_central=central[lcdm_valid]

lcdm_central_group=data["SubhaloGroupNr"][lcdm_valid][lcdm_central]
lcdm_MUV_central=lcdm_MUV[lcdm_central]
lcdm_sfr_central=lcdm_sfr[lcdm_central]
lcdm_positions_central=lcdm_positions_cmpc[lcdm_central]

lcdm_halo_m200c=halos["Group_M_Crit200"][lcdm_central_group]*1e10/lcdm_data["h"]
lcdm_halo_mvir=halos["Group_M_TopHat200"][lcdm_central_group]*1e10/lcdm_data["h"]

print("UV valid subhalos:",len(lcdm_MUV))
print("central subhalos:",np.count_nonzero(lcdm_central))
print("satellite subhalos:",np.count_nonzero(~lcdm_central))

def compute_sfe(Mhalo,cap=True): #extrapolate
    Mhalo=np.asarray(Mhalo,dtype=float)
    Mhalo_model=Mhalo.copy()
    if cap:
        Mhalo_model=np.minimum(Mhalo_model,10**10.23)
    sfe=np.full(Mhalo.shape,np.nan,dtype=float)
    valid=np.isfinite(Mhalo_model)&(Mhalo_model>0)
    sfe[valid]=10**(-1.78)/((Mhalo_model[valid]/1e9)**(-0.64)+(Mhalo_model[valid]/1e9)**(-0.64)) #fitted parameters
    return sfe

#yung halo accretion model
def yung_dmdt(Mhalo,redshift,cosmology):
    Mhalo=np.asarray(Mhalo,dtype=float)
    halo_accretion=np.full(Mhalo.shape,np.nan,dtype=float)
    valid=np.isfinite(Mhalo)&(Mhalo>0)
    a=1/(1+redshift)
    alpha=0.858+1.554*a-1.176*a**2
    beta=10**(2.578-0.989*a-1.545*a**2)
    Ez=cosmology.efunc(redshift)
    halo_accretion[valid]=beta*((Mhalo[valid]/1e12)*Ez)**alpha #match index with only valid indices
    return halo_accretion

#Lumina SFR <-> mass
#copied from my HMF script
def assign_halos_to_mass_bins(m200c_msun, inside, mass_bin_width):
    masses_inside=m200c_msun[inside]
    masses_valid=(masses_inside>0)&np.isfinite(masses_inside)
    masses_inside=masses_inside[masses_valid]
    log_mass = np.log10(masses_inside)

    log_mass_min = np.floor(log_mass.min() / mass_bin_width) * mass_bin_width #cleaner, integer edges
    log_mass_max=(np.floor(log_mass.max()/mass_bin_width)+1)*mass_bin_width

    mass_bin_edges = np.arange(log_mass_min,log_mass_max + mass_bin_width,mass_bin_width,)
    n_mass_bins = len(mass_bin_edges) - 1 #total bins. Must be 1 less than the number of edges
    mass_bin_ids = np.full(len(log_mass), -1, dtype=int) #all mass bin ids of halos start with -1, meaning unassigned
    
    #goes through each bin and find all halo masses in this bin instead of the opposite more better efficiency
    for bin_id in range(n_mass_bins):
        lower_edge = mass_bin_edges[bin_id]
        upper_edge = mass_bin_edges[bin_id + 1]
        in_this_bin = ((log_mass >= lower_edge)& (log_mass < upper_edge))
        mass_bin_ids[in_this_bin] = bin_id
 
    return mass_bin_ids, mass_bin_edges, masses_valid

In [ ]:
#inside the covered volume after assigning fields
lcdm_central_inside=lcdm_inside[lcdm_central]
lcdm_mass_bin_ids,lcdm_mass_edges,lcdm_in_mass_range=assign_halos_to_mass_bins(m200c_msun=lcdm_halo_m200c,inside=lcdm_central_inside,mass_bin_width=0.3)
lcdm_mass_centers=0.5*(lcdm_mass_edges[:-1]+lcdm_mass_edges[1:])
lcdm_MUV_central_inside=lcdm_MUV_central[lcdm_central_inside][lcdm_in_mass_range]
lcdm_sfr_central_inside=lcdm_sfr_central[lcdm_central_inside][lcdm_in_mass_range]
lcdm_halo_m200c_inside=lcdm_halo_m200c[lcdm_central_inside][lcdm_in_mass_range]
lcdm_halo_mvir_inside=lcdm_halo_mvir[lcdm_central_inside][lcdm_in_mass_range]

lcdm_n_mass_bins=len(lcdm_mass_edges)-1
lcdm_median_sfr=np.full(lcdm_n_mass_bins,np.nan)
lcdm_per_mass_bin=np.zeros(lcdm_n_mass_bins,dtype=int)

for bin_id in range(lcdm_n_mass_bins):
    assert np.all(lcdm_mass_bin_ids>=0)
    in_this_bin=lcdm_mass_bin_ids==bin_id
    sfr_valid=in_this_bin&np.isfinite(lcdm_sfr_central_inside)
    lcdm_per_mass_bin[bin_id]=np.count_nonzero(sfr_valid)

    if lcdm_per_mass_bin[bin_id]>0: #preventing counting an empty bin
        lcdm_median_sfr[bin_id]=np.median(lcdm_sfr_central_inside[sfr_valid]) #median of this bin

for bin_id in range(lcdm_n_mass_bins):
    print(lcdm_mass_edges[bin_id],lcdm_mass_edges[bin_id+1],lcdm_per_mass_bin[bin_id],lcdm_median_sfr[bin_id])

print("number of central galaxies:",len(lcdm_sfr_central_inside))
print("number of zero sfr central galaxies:",np.count_nonzero(lcdm_sfr_central_inside<=0))

In [ ]:
#now we calculate the target SFR and divide by the median values to get the boost
lcdm_fb=cv.lumina_cosmology.Ob0/cv.lumina_cosmology.Om0
lcdm_dmdt=yung_dmdt(lcdm_halo_mvir_inside,lcdm_data["redshift"],cv.lumina_cosmology)
lcdm_galaxy_sfr=lcdm_median_sfr[lcdm_mass_bin_ids] #assign each galaxy the median sfr according to which bin it belongs to so we can match length

#comparison between extrapolating beyond 10^10.23 vs not
lcdm_sfe_capped=compute_sfe(lcdm_halo_m200c_inside,cap=True)
lcdm_sfe_extrapolated=compute_sfe(lcdm_halo_m200c_inside,cap=False)

lcdm_target_sfr_capped=lcdm_sfe_capped*lcdm_fb*lcdm_dmdt
lcdm_target_sfr_extrapolated=lcdm_sfe_extrapolated*lcdm_fb*lcdm_dmdt

sfr_valid=np.isfinite(lcdm_galaxy_sfr)&(lcdm_galaxy_sfr>0)&np.isfinite(lcdm_target_sfr_capped)&(lcdm_target_sfr_capped>0)&np.isfinite(lcdm_target_sfr_extrapolated)&(lcdm_target_sfr_extrapolated>0)

lcdm_boost_capped=np.full(len(lcdm_target_sfr_capped),np.nan)
lcdm_boost_extrapolated=np.full(len(lcdm_target_sfr_extrapolated),np.nan)

lcdm_boost_capped[sfr_valid]=lcdm_target_sfr_capped[sfr_valid]/lcdm_galaxy_sfr[sfr_valid]
lcdm_boost_extrapolated[sfr_valid]=lcdm_target_sfr_extrapolated[sfr_valid]/lcdm_galaxy_sfr[sfr_valid]

lcdm_MUV_shen_capped=np.full(len(lcdm_MUV_central_inside),np.nan)
lcdm_MUV_shen_extrapolated=np.full(len(lcdm_MUV_central_inside),np.nan)

lcdm_MUV_shen_capped[sfr_valid]=lcdm_MUV_central_inside[sfr_valid]-2.5*np.log10(lcdm_boost_capped[sfr_valid])
lcdm_MUV_shen_extrapolated[sfr_valid]=lcdm_MUV_central_inside[sfr_valid]-2.5*np.log10(lcdm_boost_extrapolated[sfr_valid])

print("central galaxies modeled:",np.count_nonzero(sfr_valid))
print("central galaxies not modeled:",len(lcdm_MUV_central_inside)-np.count_nonzero(sfr_valid)) #counts the number of nonozeros, or True (=1) elements, which are used and passed the mask
print("modeled galaxies above Shen calibration:",np.count_nonzero(sfr_valid&(lcdm_halo_m200c_inside>10**10.23)))
print("capped SFE range:",np.nanmin(lcdm_sfe_capped[sfr_valid]),np.nanmax(lcdm_sfe_capped[sfr_valid]))
print("extrapolated SFE range:",np.nanmin(lcdm_sfe_extrapolated[sfr_valid]),np.nanmax(lcdm_sfe_extrapolated[sfr_valid]))
print("capped boost range:",np.nanmin(lcdm_boost_capped[sfr_valid]),np.nanmax(lcdm_boost_capped[sfr_valid]))
print("extrapolated boost range:",np.nanmin(lcdm_boost_extrapolated[sfr_valid]),np.nanmax(lcdm_boost_extrapolated[sfr_valid]))

In [ ]:
lcdm_target_sfr_capped_median=np.full(lcdm_n_mass_bins,np.nan)
lcdm_target_sfr_extrapolated_median=np.full(lcdm_n_mass_bins,np.nan)
lcdm_boost_capped_median=np.full(lcdm_n_mass_bins,np.nan)
lcdm_boost_extrapolated_median=np.full(lcdm_n_mass_bins,np.nan)

for bin_id in range(lcdm_n_mass_bins):
    in_this_bin=lcdm_mass_bin_ids==bin_id

    if np.any(in_this_bin):
        lcdm_target_sfr_capped_median[bin_id]=np.median(lcdm_target_sfr_capped[in_this_bin])
        lcdm_target_sfr_extrapolated_median[bin_id]=np.median(lcdm_target_sfr_extrapolated[in_this_bin])
        lcdm_boost_capped_median[bin_id]=np.median(lcdm_boost_capped[in_this_bin])
        lcdm_boost_extrapolated_median[bin_id]=np.median(lcdm_boost_extrapolated[in_this_bin])
        
lcdm_plot_valid=np.isfinite(lcdm_median_sfr)&(lcdm_median_sfr>0)&np.isfinite(lcdm_target_sfr_capped_median)&np.isfinite(lcdm_target_sfr_extrapolated_median)
lcdm_plot_mass=10**lcdm_mass_centers[lcdm_plot_valid]

fig,ax=plt.subplots(1,2,figsize=(14,5),facecolor="white")
    
ax[0].set_facecolor("white")
ax[1].set_facecolor("white")

ax[0].plot(lcdm_plot_mass,lcdm_median_sfr[lcdm_plot_valid],marker="o",color="black",label="Lumina median")
ax[0].plot(lcdm_plot_mass,lcdm_target_sfr_capped_median[lcdm_plot_valid],marker="o",color="blue",label="Shen+26 capped")
ax[0].plot(lcdm_plot_mass,lcdm_target_sfr_extrapolated_median[lcdm_plot_valid],marker="o",color="purple",label="Shen+26 extrapolated")
ax[0].axvline(10**10.23,color="gray",linestyle=":",label="Shen calibration limit")
ax[0].set_xscale("log")
ax[0].set_yscale("log")
ax[0].set_xlabel(r"$M_{200c}\,[M_\odot]$")
ax[0].set_ylabel(r"${\rm SFR}\,[M_\odot\,{\rm yr}^{-1}]$")
ax[0].legend()

ax[1].plot(lcdm_plot_mass,lcdm_boost_capped_median[lcdm_plot_valid],marker="o",color="blue",label="Capped")
ax[1].plot(lcdm_plot_mass,lcdm_boost_extrapolated_median[lcdm_plot_valid],marker="o",color="purple",label="Extrapolated")
ax[1].axhline(1,color="black",linestyle="--")
ax[1].axvline(10**10.23,color="gray",linestyle=":")
ax[1].set_xscale("log")
ax[1].set_yscale("log")
ax[1].set_xlabel(r"$M_{200c}\,[M_\odot]$")
ax[1].set_ylabel(r"$B_{\rm SFR}$")
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
lcdm_log_mass_grid=np.linspace(8,12.2,400)
lcdm_mass_grid=10**lcdm_log_mass_grid

lcdm_dmdt_grid=yung_dmdt(lcdm_mass_grid,10,cv.lumina_cosmology)
lcdm_sfe_grid_capped=compute_sfe(lcdm_mass_grid,cap=True)
lcdm_sfe_grid_extrapolated=compute_sfe(lcdm_mass_grid,cap=False)

lcdm_dmdt_slope=np.polyfit(np.log10(lcdm_mass_grid),np.log10(lcdm_dmdt_grid),1)[0]
lcdm_sfe_extrapolated_slope=np.polyfit(np.log10(lcdm_mass_grid),np.log10(lcdm_sfe_grid_extrapolated),1)[0]

lcdm_above_calibration=lcdm_mass_grid>10**10.23
lcdm_sfe_capped_slope=np.polyfit(np.log10(lcdm_mass_grid[lcdm_above_calibration]),np.log10(lcdm_sfe_grid_capped[lcdm_above_calibration]),1)[0]

fig,ax=plt.subplots(1,2,figsize=(14,5),facecolor="white")
ax[0].set_facecolor("white")
ax[1].set_facecolor("white")

ax[0].plot(lcdm_mass_grid,lcdm_dmdt_grid,color="green",linewidth=2)
ax[0].set_xscale("log")
ax[0].set_yscale("log")
ax[0].set_xlabel(r"$M_{\rm vir}\,[M_\odot]$")
ax[0].set_ylabel(r"$\dot M_{\rm vir}\,[M_\odot\,{\rm yr}^{-1}]$")
ax[0].text(0.05,0.9,rf"$\dot M_{{\rm vir}}\propto M_{{\rm vir}}^{{{lcdm_dmdt_slope:.2f}}}$",transform=ax[0].transAxes)

ax[1].plot(lcdm_log_mass_grid,lcdm_sfe_grid_capped,color="blue",linewidth=2,label="Capped")
ax[1].plot(lcdm_log_mass_grid,lcdm_sfe_grid_extrapolated,color="purple",linewidth=2,label="Extrapolated")
ax[1].axvline(10.23,color="gray",linestyle=":")

ax[1].set_yscale("log")
ax[1].set_xlim(8,12.2)
ax[1].set_xticks(np.arange(8,12.1,0.5))
ax[1].set_xlabel(r"$\log_{10}(M_{\rm halo}/M_\odot)$")
ax[1].set_ylabel(r"$\epsilon_{\rm halo}$")
ax[1].legend()

ax[0].tick_params(which="both",direction="in",top=True,right=True)
ax[1].tick_params(which="both",direction="in",top=True,right=True)

plt.tight_layout()
plt.show()

print("mass-accretion slope:",lcdm_dmdt_slope)
print("extrapolated SFE slope:",lcdm_sfe_extrapolated_slope)
print("capped SFE slope above calibration:",lcdm_sfe_capped_slope)

In [ ]:
#CV of current SFR-ratio model
lcdm_central_mask=lcdm_central[lcdm_inside]
lcdm_field_ids_central=lcdm_field_ids[lcdm_central_mask][lcdm_in_mass_range]
# lcdm_halo_mh=lcdm_halo_m200c_inside*lcdm_data["h"]
lcdm_subhalo_mass_central=lcdm_subhalo_mass_inside[lcdm_central_mask][lcdm_in_mass_range]

lcdm_sigma_m=cv.compute_sigma_m(cosmology=cv.which_cosmology("LCDM"),field_lengths_cmpc=lcdm_field_lengths_cmpc,box_size_cmpc=lcdm_data["box_size_cmpc"],h=lcdm_data["h"],redshift=lcdm_data["redshift"])

lcdm_shen_capped_results=cv.og_UV_model(MUV_inside=lcdm_MUV_shen_capped,subhalo_mass_inside=lcdm_subhalo_mass_central,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids_central,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Shen capped")
lcdm_shen_extrapolated_results=cv.og_UV_model(MUV_inside=lcdm_MUV_shen_extrapolated,subhalo_mass_inside=lcdm_subhalo_mass_central,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids_central,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Shen extrapolated")

print("model MUV_limit mu sigma_measured sigma_linear ratio total")
for model_results in [lcdm_shen_capped_results,lcdm_shen_extrapolated_results]:
    for result in model_results:
        result["sigma_m"]=lcdm_sigma_m
        result["sigma_cv_predicted"]=result["b_eff"]*lcdm_sigma_m
        result["cv_ratio"]=result["sigma_cv"]/result["sigma_cv_predicted"]
        print(result["model"],result["MUV_limit"],result["mu"],result["sigma_cv"],result["sigma_cv_predicted"],result["cv_ratio"],result["total_galaxies"])

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(14,5),facecolor="white")
ax[0].set_facecolor("white")
ax[1].set_facecolor("white")

for model_results,label,color,marker in [(lcdm_shen_capped_results,"lumina capped","blue","o"),(lcdm_shen_extrapolated_results,"lumina extrapolated","purple","s")]:
    MUV_limit=np.array([result["MUV_limit"] for result in model_results])
    sigma_measured=np.array([result["sigma_cv"] for result in model_results])
    sigma_predicted=np.array([result["b_eff"]*lcdm_sigma_m for result in model_results])
    error_low=np.array([result["sigma_cv_err_low"] for result in model_results])
    error_high=np.array([result["sigma_cv_err_high"] for result in model_results])
    ratio=sigma_measured/sigma_predicted

    ax[0].errorbar(MUV_limit,sigma_measured,yerr=[error_low,error_high],marker=marker,color=color,linestyle="-",capsize=4,label=label+" measured")
    ax[0].plot(MUV_limit,sigma_predicted,color=color,linestyle="--",label=label+r" $b_{\rm eff}\sigma_m$")

    ax[1].errorbar(MUV_limit,ratio,yerr=[error_low/sigma_predicted,error_high/sigma_predicted],marker=marker,color=color,linestyle="-",capsize=4,label=label)

ax[0].errorbar(cv.obs_MUV_limits,cv.obs_sigma_cv,yerr=[cv.obs_lower_error,cv.obs_upper_error],marker="*",markersize=14,color="red",linestyle="none",capsize=5,label="PANORAMIC obs")

ax[0].set_xlabel(r"$M_{\rm UV}\ {\rm limit}$")
ax[0].set_ylabel(r"$\sigma_{\rm CV}$")
ax[0].set_xlim(-19.3,-20.7)
ax[0].set_ylim(bottom=0)
ax[0].grid(alpha=0.3)
ax[0].legend()

ax[1].axhline(1,color="black",linestyle="--")
ax[1].set_xlabel(r"$M_{\rm UV}\ {\rm limit}$")
ax[1].set_ylabel(r"$\sigma_{\rm CV}^{\rm measured}/(b_{\rm eff}\sigma_m)$")
ax[1].set_xlim(-19.3,-20.7)
ax[1].grid(alpha=0.3)
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
#UVLF satellite vs central galaxies
#og=og values of centrals in the volume produced by lumina
lcdm_central_mask=lcdm_central[lcdm_inside]
lcdm_satellite_mask=~lcdm_central_mask

lcdm_MUV_central_og_inside=lcdm_MUV_inside[lcdm_central_mask]
lcdm_MUV_satellite_inside=lcdm_MUV_inside[lcdm_satellite_mask]

lcdm_subhalo_mass_central_og_inside=lcdm_subhalo_mass_inside[lcdm_central_mask]
lcdm_subhalo_mass_satellite_inside=lcdm_subhalo_mass_inside[lcdm_satellite_mask]

lcdm_field_ids_central_og=lcdm_field_ids[lcdm_central_mask]
lcdm_field_ids_satellite=lcdm_field_ids[lcdm_satellite_mask]

assert len(lcdm_MUV_central_og_inside)==len(lcdm_field_ids_central_og)
assert len(lcdm_MUV_satellite_inside)==len(lcdm_field_ids_satellite)
assert len(lcdm_MUV_inside)==len(lcdm_MUV_central_og_inside)+len(lcdm_MUV_satellite_inside)

print("all subhalos inside:",len(lcdm_MUV_inside))
print("centrals inside:",len(lcdm_MUV_central_og_inside))
print("satellites inside:",len(lcdm_MUV_satellite_inside))
print("satellite fraction:",len(lcdm_MUV_satellite_inside)/len(lcdm_MUV_inside))

In [ ]:
#not Shen model modified
lcdm_MUV_central_bin_centers,lcdm_central_log_phi=cv.compute_UVLF(MUV_values=lcdm_MUV_central_og_inside,field_volume=lcdm_field_volume,n_fields=lcdm_n_fields,MUV_bin_edges=cv.MUV_bin_edges,MUV_bin_width=cv.MUV_bin_width)
lcdm_MUV_satellite_bin_centers,lcdm_satellite_log_phi=cv.compute_UVLF(MUV_values=lcdm_MUV_satellite_inside,field_volume=lcdm_field_volume,n_fields=lcdm_n_fields,MUV_bin_edges=cv.MUV_bin_edges,MUV_bin_width=cv.MUV_bin_width)

fig,ax=plt.subplots(figsize=(9,7))
ax.plot(lcdm_MUV_bin_centers,lcdm_log_phi,marker="s",color="gray",label="all subhalos")
ax.plot(lcdm_MUV_central_bin_centers,lcdm_central_log_phi,marker="o",color="blue",label="centrals")
ax.plot(lcdm_MUV_satellite_bin_centers,lcdm_satellite_log_phi,marker="^",color="orange",label="satellites")

cv.obs.plot_photoz_constraints(redshift=10,ax=ax,marker="o",ms=5,linestyle="none",color="black",capsize=4)
cv.obs.plot_specz_constraints(redshift=10,ax=ax,marker="s",ms=5,linestyle="none",color="red",capsize=4)

ax.set_xlabel(r"$M_{\rm UV}$")
ax.set_ylabel(r"$\log_{10}(\Phi/{\rm Mpc}^{-3}\,{\rm mag}^{-1})$")
ax.text(0.9,0.9,r"$z\simeq10$",transform=ax.transAxes,ha="right",va="top",fontsize=25)
ax.set_xlim(-16,-24)
ax.set_ylim(-7.5,0.5)
ax.grid(alpha=0.3)
ax.legend(fontsize=15,markerscale=0.6,handlelength=1.5,labelspacing=0.4,borderpad=0.7,frameon=True)
plt.show()

In [ ]:
#shen model modified comparison between central vs satellite. basically redoing what was done previously on centrals now to all galaxies
#assign every inside galaxy to its host halo
lcdm_central_mask=lcdm_central[lcdm_inside]
lcdm_group_inside=data["SubhaloGroupNr"][lcdm_valid][lcdm_inside]
lcdm_host_m200c_inside=halos["Group_M_Crit200"][lcdm_group_inside]*1e10/lcdm_data["h"] #every subhalo with the same group index gets the same M200c host mass
lcdm_host_mvir_inside=halos["Group_M_TopHat200"][lcdm_group_inside]*1e10/lcdm_data["h"]

#assign every host mass to the existing central-galaxy mass bins
lcdm_mass_bin_ids_all=np.full(len(lcdm_MUV_inside),-1,dtype=int)
lcdm_host_mass_valid=np.isfinite(lcdm_host_m200c_inside)&(lcdm_host_m200c_inside>0)&np.isfinite(lcdm_host_mvir_inside)&(lcdm_host_mvir_inside>0)
lcdm_mass_bin_ids_all[lcdm_host_mass_valid]=np.digitize(np.log10(lcdm_host_m200c_inside[lcdm_host_mass_valid]),lcdm_mass_edges)-1
lcdm_in_mass_range_all=lcdm_host_mass_valid&(lcdm_mass_bin_ids_all>=0)&(lcdm_mass_bin_ids_all<lcdm_n_mass_bins)

#give every galaxy the original median SFR of its host-mass bin
lcdm_galaxy_sfr_all=np.full(len(lcdm_MUV_inside),np.nan)
lcdm_galaxy_sfr_all[lcdm_in_mass_range_all]=lcdm_median_sfr[lcdm_mass_bin_ids_all[lcdm_in_mass_range_all]]

#calculate the Shen target SFR from each galaxy's host halo
lcdm_dmdt_all=yung_dmdt(lcdm_host_mvir_inside,lcdm_data["redshift"],cv.lumina_cosmology)
lcdm_sfe_capped_all=compute_sfe(lcdm_host_m200c_inside,cap=True)
lcdm_sfe_extrapolated_all=compute_sfe(lcdm_host_m200c_inside,cap=False)
lcdm_target_sfr_capped_all=lcdm_sfe_capped_all*lcdm_fb*lcdm_dmdt_all
lcdm_target_sfr_extrapolated_all=lcdm_sfe_extrapolated_all*lcdm_fb*lcdm_dmdt_all

#calculate the relative boost for every galaxy
lcdm_shen_all_valid=lcdm_in_mass_range_all&np.isfinite(lcdm_galaxy_sfr_all)&(lcdm_galaxy_sfr_all>0)&np.isfinite(lcdm_target_sfr_capped_all)&(lcdm_target_sfr_capped_all>0)&np.isfinite(lcdm_target_sfr_extrapolated_all)&(lcdm_target_sfr_extrapolated_all>0)

lcdm_boost_capped_all=np.full(len(lcdm_MUV_inside),np.nan)
lcdm_boost_extrapolated_all=np.full(len(lcdm_MUV_inside),np.nan)
lcdm_boost_capped_all[lcdm_shen_all_valid]=lcdm_target_sfr_capped_all[lcdm_shen_all_valid]/lcdm_galaxy_sfr_all[lcdm_shen_all_valid]
lcdm_boost_extrapolated_all[lcdm_shen_all_valid]=lcdm_target_sfr_extrapolated_all[lcdm_shen_all_valid]/lcdm_galaxy_sfr_all[lcdm_shen_all_valid]

#translate each relative SFR boost into a Muv shift
lcdm_MUV_shen_capped_all=np.full(len(lcdm_MUV_inside),np.nan)
lcdm_MUV_shen_extrapolated_all=np.full(len(lcdm_MUV_inside),np.nan)
lcdm_MUV_shen_capped_all[lcdm_shen_all_valid]=lcdm_MUV_inside[lcdm_shen_all_valid]-2.5*np.log10(lcdm_boost_capped_all[lcdm_shen_all_valid])
lcdm_MUV_shen_extrapolated_all[lcdm_shen_all_valid]=lcdm_MUV_inside[lcdm_shen_all_valid]-2.5*np.log10(lcdm_boost_extrapolated_all[lcdm_shen_all_valid])

print("all galaxies modeled:",np.count_nonzero(lcdm_shen_all_valid))
print("centrals modeled:",np.count_nonzero(lcdm_shen_all_valid&lcdm_central_mask))
print("satellites modeled:",np.count_nonzero(lcdm_shen_all_valid&~lcdm_central_mask))
print("host masses outside bins:",np.count_nonzero(lcdm_host_mass_valid&~lcdm_in_mass_range_all))

In [ ]:
lcdm_field_ids_central_shen=lcdm_field_ids[lcdm_central_mask][lcdm_in_mass_range]
lcdm_subhalo_mass_central_shen=lcdm_subhalo_mass_inside[lcdm_central_mask][lcdm_in_mass_range]

lcdm_sigma_m=cv.compute_sigma_m(cosmology=cv.which_cosmology("LCDM"),field_lengths_cmpc=lcdm_field_lengths_cmpc,box_size_cmpc=lcdm_data["box_size_cmpc"],h=lcdm_data["h"],redshift=lcdm_data["redshift"])

lcdm_shen_capped_central_results=cv.og_UV_model(MUV_inside=lcdm_MUV_shen_capped,subhalo_mass_inside=lcdm_subhalo_mass_central_shen,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids_central_shen,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Capped centrals")
lcdm_shen_capped_all_results=cv.og_UV_model(MUV_inside=lcdm_MUV_shen_capped_all,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Capped all")
lcdm_shen_extrapolated_central_results=cv.og_UV_model(MUV_inside=lcdm_MUV_shen_extrapolated,subhalo_mass_inside=lcdm_subhalo_mass_central_shen,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids_central_shen,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Extrapolated centrals")
lcdm_shen_extrapolated_all_results=cv.og_UV_model(MUV_inside=lcdm_MUV_shen_extrapolated_all,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Extrapolated all")

for model_results in [lcdm_shen_capped_central_results,lcdm_shen_capped_all_results,lcdm_shen_extrapolated_central_results,lcdm_shen_extrapolated_all_results]:
    for result in model_results:
        result["sigma_cv_predicted"]=result["b_eff"]*lcdm_sigma_m
        result["cv_ratio"]=result["sigma_cv"]/result["sigma_cv_predicted"]

In [ ]:
#UVLF comparison
lcdm_MUV_shen_capped_central_bins,lcdm_log_phi_shen_capped_central=cv.compute_UVLF(MUV_values=lcdm_MUV_shen_capped,field_volume=lcdm_field_volume,n_fields=lcdm_n_fields,MUV_bin_edges=cv.MUV_bin_edges,MUV_bin_width=cv.MUV_bin_width)
lcdm_MUV_shen_capped_all_bins,lcdm_log_phi_shen_capped_all=cv.compute_UVLF(MUV_values=lcdm_MUV_shen_capped_all,field_volume=lcdm_field_volume,n_fields=lcdm_n_fields,MUV_bin_edges=cv.MUV_bin_edges,MUV_bin_width=cv.MUV_bin_width)
lcdm_MUV_shen_extrapolated_central_bins,lcdm_log_phi_shen_extrapolated_central=cv.compute_UVLF(MUV_values=lcdm_MUV_shen_extrapolated,field_volume=lcdm_field_volume,n_fields=lcdm_n_fields,MUV_bin_edges=cv.MUV_bin_edges,MUV_bin_width=cv.MUV_bin_width)
lcdm_MUV_shen_extrapolated_all_bins,lcdm_log_phi_shen_extrapolated_all=cv.compute_UVLF(MUV_values=lcdm_MUV_shen_extrapolated_all,field_volume=lcdm_field_volume,n_fields=lcdm_n_fields,MUV_bin_edges=cv.MUV_bin_edges,MUV_bin_width=cv.MUV_bin_width)

fig,ax=plt.subplots(1,2,figsize=(14,6),facecolor="white")

ax[0].plot(lcdm_MUV_bin_centers,lcdm_log_phi,color="gray",marker="s",label="Lumina")
ax[0].plot(lcdm_MUV_shen_capped_central_bins,lcdm_log_phi_shen_capped_central,color="blue",marker="o",linestyle="--",label="Centrals")
ax[0].plot(lcdm_MUV_shen_capped_all_bins,lcdm_log_phi_shen_capped_all,color="orange",marker="^",label="All galaxies")
ax[0].set_title("Capped")

ax[1].plot(lcdm_MUV_bin_centers,lcdm_log_phi,color="gray",marker="s",label="Lumina")
ax[1].plot(lcdm_MUV_shen_extrapolated_central_bins,lcdm_log_phi_shen_extrapolated_central,color="blue",marker="o",linestyle="--",label="Centrals")
ax[1].plot(lcdm_MUV_shen_extrapolated_all_bins,lcdm_log_phi_shen_extrapolated_all,color="orange",marker="^",label="All galaxies")
ax[1].set_title("Extrapolated")

for axis in ax:
    cv.obs.plot_photoz_constraints(redshift=10,ax=axis,marker="o",ms=5,linestyle="none",color="black",capsize=4)
    cv.obs.plot_specz_constraints(redshift=10,ax=axis,marker="s",ms=5,linestyle="none",color="red",capsize=4)
    axis.set_xlabel(r"$M_{\rm UV}$")
    axis.set_ylabel(r"$\log_{10}(\Phi/{\rm Mpc}^{-3}\,{\rm mag}^{-1})$")
    axis.set_xlim(-16,-24)
    axis.set_ylim(-7.5,-2)
    axis.text(0.9, 0.9,r'$z\simeq10$',transform=axis.transAxes,ha='right',va='top',fontsize=25)
    axis.grid(alpha=0.3)
    axis.legend()

plt.tight_layout()
plt.show()

In [ ]:
cv_panels=[(lcdm_shen_capped_central_results,lcdm_shen_capped_all_results,"Capped"),(lcdm_shen_extrapolated_central_results,lcdm_shen_extrapolated_all_results,"Extrapolated")]

fig,ax=plt.subplots(1,2,figsize=(14,6),facecolor="white")

for panel_id,(central_results,all_results,title) in enumerate(cv_panels):
    for model_results,label,color,marker in [(central_results,"Centrals","blue","o"),(all_results,"All galaxies","orange","^")]:
        MUV_limit=np.array([result["MUV_limit"] for result in model_results])
        sigma_measured=np.array([result["sigma_cv"] for result in model_results])
        sigma_predicted=np.array([result["sigma_cv_predicted"] for result in model_results])
        error_low=np.array([result["sigma_cv_err_low"] for result in model_results])
        error_high=np.array([result["sigma_cv_err_high"] for result in model_results])

        ax[panel_id].errorbar(MUV_limit,sigma_measured,yerr=[error_low,error_high],color=color,marker=marker,linestyle="-",capsize=4,label=label+" measured")
        ax[panel_id].plot(MUV_limit,sigma_predicted,color=color,linestyle="--",label=label+r" $b_{\rm eff}\sigma_m$")

    ax[panel_id].errorbar(cv.obs_MUV_limits,cv.obs_sigma_cv,yerr=[cv.obs_lower_error,cv.obs_upper_error],color="red",marker="*",markersize=14,linestyle="none",capsize=5,label="PANORAMIC")
    ax[panel_id].set_title(title)
    ax[panel_id].set_xlabel(r"$M_{\rm UV}\ {\rm limit}$")
    ax[panel_id].set_ylabel(r"$\sigma_{\rm CV}$")
    ax[panel_id].set_xlim(-19.3,-20.7)
    ax[panel_id].set_ylim(bottom=0)
    ax[panel_id].grid(alpha=0.3)
    ax[panel_id].text(0.85, 0.9,r'$z\simeq10$',transform=ax[panel_id].transAxes,ha='right',va='top',fontsize=25)
    ax[panel_id].legend()

plt.tight_layout()
plt.show()

In [ ]:
#cosmic variance ratios
fig,ax=plt.subplots(1,2,figsize=(14,6),facecolor="white")

for panel_id,(central_results,all_results,title) in enumerate(cv_panels):
    for model_results,label,color,marker in [(central_results,"Centrals","blue","o"),(all_results,"All galaxies","orange","^")]:
        MUV_limit=np.array([result["MUV_limit"] for result in model_results])
        ratio=np.array([result["cv_ratio"] for result in model_results])
        sigma_predicted=np.array([result["sigma_cv_predicted"] for result in model_results])
        error_low=np.array([result["sigma_cv_err_low"] for result in model_results])/sigma_predicted
        error_high=np.array([result["sigma_cv_err_high"] for result in model_results])/sigma_predicted

        ax[panel_id].errorbar(MUV_limit,ratio,yerr=[error_low,error_high],color=color,marker=marker,linestyle="-",capsize=4,label=label)

    ax[panel_id].axhline(1,color="black",linestyle="--")
    ax[panel_id].set_title(title)
    ax[panel_id].set_xlabel(r"$M_{\rm UV}\ {\rm limit}$")
    ax[panel_id].set_ylabel(r"$\sigma_{\rm CV}^{\rm measured}/(b_{\rm eff}\sigma_m)$")
    ax[panel_id].set_xlim(-19.3,-20.7)
    ax[panel_id].text(0.84, 0.91,r'$z\simeq10$',transform=ax[panel_id].transAxes,ha='right',va='top',fontsize=25)
    ax[panel_id].grid(alpha=0.3)
    ax[panel_id].legend()

plt.tight_layout()
plt.show()

In [ ]:
def sfr_to_MUV(sfr,kappa_uv=0.72e-28):
    sfr=np.asarray(sfr,dtype=float)
    MUV=np.full(sfr.shape,np.nan)
    valid=np.isfinite(sfr)&(sfr>0) #for the analytic SFR, it must be greater than zero
    luminosity=sfr[valid]/kappa_uv
    MUV[valid]=51.60-2.5*np.log10(luminosity)
    return MUV #analytic MUV

#convert the target SFR relations into target MUV
lcdm_target_MUV_capped=sfr_to_MUV(lcdm_target_sfr_capped)
lcdm_target_MUV_extrapolated=sfr_to_MUV(lcdm_target_sfr_extrapolated)

#measure lumina's median MUV in each existing halo-mass bin
lcdm_median_MUV=np.full(lcdm_n_mass_bins,np.nan)
lcdm_target_MUV_capped_median=np.full(lcdm_n_mass_bins,np.nan)
lcdm_target_MUV_extrapolated_median=np.full(lcdm_n_mass_bins,np.nan)

for bin_id in range(lcdm_n_mass_bins): #calculation for each mass bin for delta MUV
    in_this_bin=lcdm_mass_bin_ids==bin_id
    og_valid=in_this_bin&np.isfinite(lcdm_MUV_central_inside) #data from Lumina calculated MUV
    capped_valid=in_this_bin&np.isfinite(lcdm_target_MUV_capped) #model predicted values
    extrapolated_valid=in_this_bin&np.isfinite(lcdm_target_MUV_extrapolated) #model predicted values

    if np.any(og_valid):
        lcdm_median_MUV[bin_id]=np.median(lcdm_MUV_central_inside[og_valid])
    if np.any(capped_valid):
        lcdm_target_MUV_capped_median[bin_id]=np.median(lcdm_target_MUV_capped[capped_valid])
    if np.any(extrapolated_valid):
        lcdm_target_MUV_extrapolated_median[bin_id]=np.median(lcdm_target_MUV_extrapolated[extrapolated_valid])

#find magnitude shift in each mass bin
lcdm_MUV_shift_capped_per_bin=lcdm_target_MUV_capped_median-lcdm_median_MUV
lcdm_MUV_shift_extrapolated_per_bin=lcdm_target_MUV_extrapolated_median-lcdm_median_MUV

#create output arrays for central subhalos
lcdm_MUV_luminosity_capped=np.full(len(lcdm_MUV_central_inside),np.nan)
lcdm_MUV_luminosity_extrapolated=np.full(len(lcdm_MUV_central_inside),np.nan)

#create output arrays for all subhalos
lcdm_MUV_luminosity_capped_all=np.full(len(lcdm_MUV_inside),np.nan)
lcdm_MUV_luminosity_extrapolated_all=np.full(len(lcdm_MUV_inside),np.nan)

for bin_id in range(lcdm_n_mass_bins):
    central_in_bin=(lcdm_mass_bin_ids==bin_id)&np.isfinite(lcdm_MUV_central_inside)
    all_in_bin=lcdm_in_mass_range_all&(lcdm_mass_bin_ids_all==bin_id)&np.isfinite(lcdm_MUV_inside)

    if np.isfinite(lcdm_MUV_shift_capped_per_bin[bin_id]):
        lcdm_MUV_luminosity_capped[central_in_bin]=lcdm_MUV_central_inside[central_in_bin]+lcdm_MUV_shift_capped_per_bin[bin_id]
        lcdm_MUV_luminosity_capped_all[all_in_bin]=lcdm_MUV_inside[all_in_bin]+lcdm_MUV_shift_capped_per_bin[bin_id]

    if np.isfinite(lcdm_MUV_shift_extrapolated_per_bin[bin_id]):
        lcdm_MUV_luminosity_extrapolated[central_in_bin]=lcdm_MUV_central_inside[central_in_bin]+lcdm_MUV_shift_extrapolated_per_bin[bin_id]
        lcdm_MUV_luminosity_extrapolated_all[all_in_bin]=lcdm_MUV_inside[all_in_bin]+lcdm_MUV_shift_extrapolated_per_bin[bin_id]

print("capped centrals modeled:",np.count_nonzero(np.isfinite(lcdm_MUV_luminosity_capped)))
print("extrapolated centrals modeled:",np.count_nonzero(np.isfinite(lcdm_MUV_luminosity_extrapolated)))
print("capped all galaxies modeled:",np.count_nonzero(np.isfinite(lcdm_MUV_luminosity_capped_all)))
print("extrapolated all galaxies modeled:",np.count_nonzero(np.isfinite(lcdm_MUV_luminosity_extrapolated_all)))

In [ ]:
#measure the original MUV of zero-SFR centrals and the final SFR-ratio model medians
lcdm_zero_sfr_MUV_median=np.full(lcdm_n_mass_bins,np.nan)
lcdm_sfr_native_MUV_median=np.full(lcdm_n_mass_bins,np.nan)
lcdm_sfr_capped_MUV_median=np.full(lcdm_n_mass_bins,np.nan)
lcdm_sfr_extrapolated_MUV_median=np.full(lcdm_n_mass_bins,np.nan)

for bin_id in range(lcdm_n_mass_bins):
    in_this_bin=lcdm_mass_bin_ids==bin_id

    zero_sfr_valid=in_this_bin&np.isfinite(lcdm_MUV_central_inside)&np.isfinite(lcdm_sfr_central_inside)&(lcdm_sfr_central_inside<=0)
    sfr_model_valid=in_this_bin&np.isfinite(lcdm_MUV_central_inside)&np.isfinite(lcdm_MUV_shen_capped)&np.isfinite(lcdm_MUV_shen_extrapolated)

    if np.any(zero_sfr_valid):
        lcdm_zero_sfr_MUV_median[bin_id]=np.median(lcdm_MUV_central_inside[zero_sfr_valid])

    if np.any(sfr_model_valid):
        lcdm_sfr_native_MUV_median[bin_id]=np.median(lcdm_MUV_central_inside[sfr_model_valid])
        lcdm_sfr_capped_MUV_median[bin_id]=np.median(lcdm_MUV_shen_capped[sfr_model_valid])
        lcdm_sfr_extrapolated_MUV_median[bin_id]=np.median(lcdm_MUV_shen_extrapolated[sfr_model_valid])

#measure the actual median shifts produced by the SFR-ratio method
lcdm_MUV_shift_sfr_capped_per_bin=lcdm_sfr_capped_MUV_median-lcdm_sfr_native_MUV_median
lcdm_MUV_shift_sfr_extrapolated_per_bin=lcdm_sfr_extrapolated_MUV_median-lcdm_sfr_native_MUV_median

#separate plotting masks so low-mass zero-SFR bins are not removed
native_plot_valid=np.isfinite(lcdm_median_MUV)
zero_sfr_plot_valid=np.isfinite(lcdm_zero_sfr_MUV_median)
target_capped_plot_valid=np.isfinite(lcdm_target_MUV_capped_median)
target_extrapolated_plot_valid=np.isfinite(lcdm_target_MUV_extrapolated_median)

luminosity_capped_plot_valid=np.isfinite(lcdm_MUV_shift_capped_per_bin)
luminosity_extrapolated_plot_valid=np.isfinite(lcdm_MUV_shift_extrapolated_per_bin)
sfr_capped_plot_valid=np.isfinite(lcdm_MUV_shift_sfr_capped_per_bin)
sfr_extrapolated_plot_valid=np.isfinite(lcdm_MUV_shift_sfr_extrapolated_per_bin)

fig,ax=plt.subplots(1,2,figsize=(14,5),facecolor="white")

#original and target MUV relations
ax[0].plot(lcdm_mass_centers[native_plot_valid],lcdm_median_MUV[native_plot_valid],color="black",marker="o",label="Lumina median")
ax[0].plot(lcdm_mass_centers[zero_sfr_plot_valid],lcdm_zero_sfr_MUV_median[zero_sfr_plot_valid],color="gray",marker="x",linestyle="--",label="Lumina median,SFR=0")
ax[0].plot(lcdm_mass_centers[target_capped_plot_valid],lcdm_target_MUV_capped_median[target_capped_plot_valid],color="blue",marker="o",label="Shen target,capped")
ax[0].plot(lcdm_mass_centers[target_extrapolated_plot_valid],lcdm_target_MUV_extrapolated_median[target_extrapolated_plot_valid],color="purple",marker="o",label="Shen target,extrapolated")
ax[0].axvline(9.6,color="gray",linestyle="--",label="approximate nonzero-SFR transition")
ax[0].axvline(10.23,color="gray",linestyle=":")
ax[0].invert_yaxis()
ax[0].set_xlabel(r"$\log_{10}(M_{200c}/M_\odot)$")
ax[0].set_ylabel(r"$M_{\rm UV}$")
ax[0].set_title("Lumina and target relations")
ax[0].grid(alpha=0.3)
ax[0].legend()

#compare the four model transformations
ax[1].plot(lcdm_mass_centers[luminosity_capped_plot_valid],lcdm_MUV_shift_capped_per_bin[luminosity_capped_plot_valid],color="blue",marker="o",linestyle="-",label="Luminosity,capped")
ax[1].plot(lcdm_mass_centers[luminosity_extrapolated_plot_valid],lcdm_MUV_shift_extrapolated_per_bin[luminosity_extrapolated_plot_valid],color="purple",marker="o",linestyle="-",label="Luminosity,extrapolated")
ax[1].plot(lcdm_mass_centers[sfr_capped_plot_valid],lcdm_MUV_shift_sfr_capped_per_bin[sfr_capped_plot_valid],color="blue",marker="s",linestyle="--",label="SFR ratio,capped")
ax[1].plot(lcdm_mass_centers[sfr_extrapolated_plot_valid],lcdm_MUV_shift_sfr_extrapolated_per_bin[sfr_extrapolated_plot_valid],color="purple",marker="s",linestyle="--",label="SFR ratio,extrapolated")
ax[1].axhline(0,color="black",linestyle="--")
ax[1].axvline(9.6,color="gray",linestyle="--")
ax[1].axvline(10.23,color="gray",linestyle=":")
ax[1].set_xlabel(r"$\log_{10}(M_{200c}/M_\odot)$")
ax[1].set_ylabel(r"$\Delta M_{\rm UV}$")
ax[1].set_title("Luminosity-space versus SFR-ratio shifts")
ax[1].grid(alpha=0.3)
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
#UVLF plots
lcdm_MUV_og_all=np.full(len(lcdm_MUV_inside),np.nan)
lcdm_MUV_og_all[lcdm_in_mass_range_all]=lcdm_MUV_inside[lcdm_in_mass_range_all]

central_UVLF_models=[
    (lcdm_MUV_central_inside,"Lumina original","gray","D","-"),
    (lcdm_MUV_luminosity_capped,"Luminosity,capped","blue","o","-"),
    (lcdm_MUV_luminosity_extrapolated,"Luminosity,extrapolated","purple","o","-"),
    (lcdm_MUV_shen_capped,"SFR ratio,capped","blue","s","--"),
    (lcdm_MUV_shen_extrapolated,"SFR ratio,extrapolated","purple","s","--")
]

all_UVLF_models=[
    (lcdm_MUV_og_all,"Lumina original","gray","D","-"),
    (lcdm_MUV_luminosity_capped_all,"Luminosity,capped","blue","o","-"),
    (lcdm_MUV_luminosity_extrapolated_all,"Luminosity,extrapolated","purple","o","-"),
    (lcdm_MUV_shen_capped_all,"SFR ratio,capped","blue","s","--"),
    (lcdm_MUV_shen_extrapolated_all,"SFR ratio,extrapolated","purple","s","--")
]

fig,ax=plt.subplots(1,2,figsize=(14,5),facecolor="white",sharey=True)

for MUV_values,label,color,marker,linestyle in central_UVLF_models:
    MUV_bins,log_phi=cv.compute_UVLF(MUV_values=MUV_values,field_volume=lcdm_field_volume,n_fields=lcdm_n_fields,MUV_bin_edges=cv.MUV_bin_edges,MUV_bin_width=cv.MUV_bin_width)
    ax[0].plot(MUV_bins,log_phi,color=color,marker=marker,linestyle=linestyle,label=label)

for MUV_values,label,color,marker,linestyle in all_UVLF_models:
    MUV_bins,log_phi=cv.compute_UVLF(MUV_values=MUV_values,field_volume=lcdm_field_volume,n_fields=lcdm_n_fields,MUV_bin_edges=cv.MUV_bin_edges,MUV_bin_width=cv.MUV_bin_width)
    ax[1].plot(MUV_bins,log_phi,color=color,marker=marker,linestyle=linestyle,label=label)

for plot_axis in ax:
    cv.obs.plot_photoz_constraints(redshift=10,ax=plot_axis,marker="o",ms=5,linestyle="none",color="black",capsize=4)
    cv.obs.plot_specz_constraints(redshift=10,ax=plot_axis,marker="s",ms=5,linestyle="none",color="red",capsize=4)
    plot_axis.set_xlabel(r"$M_{\rm UV}$")
    plot_axis.set_xlim(-24,-16)
    plot_axis.set_ylim(-7.5,0.5)
    plot_axis.grid(alpha=0.3)
    plot_axis.legend(fontsize=10)

ax[0].set_ylabel(r"$\log_{10}(\Phi/{\rm Mpc}^{-3}\,{\rm mag}^{-1})$")
ax[0].set_title("Central subhalos")
ax[1].set_title("All subhalos")

plt.tight_layout()
plt.show()

In [ ]:
#verify central and all-subhalo samples are coded separately
model_checks=[
    ("Lumina original",lcdm_MUV_central_inside,lcdm_MUV_og_all),
    ("Luminosity capped",lcdm_MUV_luminosity_capped,lcdm_MUV_luminosity_capped_all),
    ("Luminosity extrapolated",lcdm_MUV_luminosity_extrapolated,lcdm_MUV_luminosity_extrapolated_all),
    ("SFR capped",lcdm_MUV_shen_capped,lcdm_MUV_shen_capped_all),
    ("SFR extrapolated",lcdm_MUV_shen_extrapolated,lcdm_MUV_shen_extrapolated_all)
]

for model_name,MUV_central,MUV_all in model_checks:
    central_from_all=MUV_all[lcdm_central_mask][lcdm_in_mass_range]
    satellite_from_all=MUV_all[~lcdm_central_mask]

    assert np.allclose(central_from_all,MUV_central,equal_nan=True)

    all_counts,_=np.histogram(MUV_all,bins=cv.MUV_bin_edges)
    central_counts,_=np.histogram(MUV_all[lcdm_central_mask],bins=cv.MUV_bin_edges)
    satellite_counts,_=np.histogram(satellite_from_all,bins=cv.MUV_bin_edges)

    assert np.array_equal(all_counts,central_counts+satellite_counts)

    print("\n"+model_name)
    for MUV_limit in cv.MUV_limits:
        central_bright=np.count_nonzero(central_from_all<MUV_limit)
        satellite_bright=np.count_nonzero(satellite_from_all<MUV_limit)
        all_bright=np.count_nonzero(MUV_all<MUV_limit)

        assert all_bright==central_bright+satellite_bright

        satellite_fraction=satellite_bright/all_bright if all_bright>0 else np.nan
        difference_dex=np.log10(all_bright/central_bright) if central_bright>0 else np.nan

        print(MUV_limit,central_bright,satellite_bright,all_bright,satellite_fraction,difference_dex)

In [ ]:
#measured cv comparison
lcdm_field_ids_model_central=lcdm_field_ids[lcdm_central_mask][lcdm_in_mass_range]
lcdm_bias_mass_model_central=lcdm_subhalo_mass_inside[lcdm_central_mask][lcdm_in_mass_range]

lcdm_luminosity_capped_central_results=cv.og_UV_model(MUV_inside=lcdm_MUV_luminosity_capped,subhalo_mass_inside=lcdm_bias_mass_model_central,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids_model_central,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Luminosity capped,centrals")

lcdm_luminosity_extrapolated_central_results=cv.og_UV_model(MUV_inside=lcdm_MUV_luminosity_extrapolated,subhalo_mass_inside=lcdm_bias_mass_model_central,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids_model_central,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Luminosity extrapolated,centrals")

lcdm_sfr_capped_central_results=cv.og_UV_model(MUV_inside=lcdm_MUV_shen_capped,subhalo_mass_inside=lcdm_bias_mass_model_central,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids_model_central,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="SFR capped,centrals")

lcdm_sfr_extrapolated_central_results=cv.og_UV_model(MUV_inside=lcdm_MUV_shen_extrapolated,subhalo_mass_inside=lcdm_bias_mass_model_central,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids_model_central,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="SFR extrapolated,centrals")

lcdm_luminosity_capped_all_results=cv.og_UV_model(MUV_inside=lcdm_MUV_luminosity_capped_all,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Luminosity capped,all")

lcdm_luminosity_extrapolated_all_results=cv.og_UV_model(MUV_inside=lcdm_MUV_luminosity_extrapolated_all,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Luminosity extrapolated,all")

lcdm_sfr_capped_all_results=cv.og_UV_model(MUV_inside=lcdm_MUV_shen_capped_all,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="SFR capped,all")

lcdm_sfr_extrapolated_all_results=cv.og_UV_model(MUV_inside=lcdm_MUV_shen_extrapolated_all,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="SFR extrapolated,all")

lcdm_lumina_central_results=cv.og_UV_model(MUV_inside=lcdm_MUV_central_inside,subhalo_mass_inside=lcdm_subhalo_mass_central_shen,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids_central_shen,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="Lumina original central galaxies")

lcdm_lumina_all_results=cv.og_UV_model(MUV_inside=lcdm_MUV_og_all,subhalo_mass_inside=lcdm_subhalo_mass_inside,redshift=lcdm_data["redshift"],field_ids=lcdm_field_ids,n_fields=lcdm_n_fields,MUV_limits=cv.MUV_limits,model_name="LUMINA original all galaxies")

all_model_results=[
    lcdm_lumina_central_results,
    lcdm_lumina_all_results,
    lcdm_luminosity_capped_central_results,
    lcdm_luminosity_extrapolated_central_results,
    lcdm_sfr_capped_central_results,
    lcdm_sfr_extrapolated_central_results,
    lcdm_luminosity_capped_all_results,
    lcdm_luminosity_extrapolated_all_results,
    lcdm_sfr_capped_all_results,
    lcdm_sfr_extrapolated_all_results
]

for model_results in all_model_results:
    for result in model_results:
        result["sigma_cv_predicted"]=result["b_eff"]*lcdm_sigma_m
        result["cv_ratio"]=result["sigma_cv"]/result["sigma_cv_predicted"]

central_CV_models=[
    (lcdm_lumina_central_results,"Lumina original","gray","D","-"),
    (lcdm_luminosity_capped_central_results,"Luminosity,capped","blue","o","-"),
    (lcdm_luminosity_extrapolated_central_results,"Luminosity,extrapolated","purple","o","-"),
    (lcdm_sfr_capped_central_results,"SFR ratio,capped","blue","s","--"),
    (lcdm_sfr_extrapolated_central_results,"SFR ratio,extrapolated","purple","s","--")
]

all_CV_models=[
    (lcdm_lumina_all_results,"Lumina original","gray","D","-"),
    (lcdm_luminosity_capped_all_results,"Luminosity,capped","blue","o","-"),
    (lcdm_luminosity_extrapolated_all_results,"Luminosity,extrapolated","purple","o","-"),
    (lcdm_sfr_capped_all_results,"SFR ratio,capped","blue","s","--"),
    (lcdm_sfr_extrapolated_all_results,"SFR ratio,extrapolated","purple","s","--")
]

fig,ax=plt.subplots(1,2,figsize=(14,5),facecolor="white",sharey=True)

for model_results,label,color,marker,linestyle in central_CV_models:
    MUV_limit=np.array([result["MUV_limit"] for result in model_results])
    sigma_cv=np.array([result["sigma_cv"] for result in model_results])
    error_low=np.array([result["sigma_cv_err_low"] for result in model_results])
    error_high=np.array([result["sigma_cv_err_high"] for result in model_results])
    ax[0].errorbar(MUV_limit,sigma_cv,yerr=[error_low,error_high],color=color,marker=marker,linestyle=linestyle,capsize=4,label=label)

for model_results,label,color,marker,linestyle in all_CV_models:
    MUV_limit=np.array([result["MUV_limit"] for result in model_results])
    sigma_cv=np.array([result["sigma_cv"] for result in model_results])
    error_low=np.array([result["sigma_cv_err_low"] for result in model_results])
    error_high=np.array([result["sigma_cv_err_high"] for result in model_results])
    ax[1].errorbar(MUV_limit,sigma_cv,yerr=[error_low,error_high],color=color,marker=marker,linestyle=linestyle,capsize=4,label=label)

for plot_axis in ax:
    plot_axis.errorbar(cv.obs_MUV_limits,cv.obs_sigma_cv,yerr=[cv.obs_lower_error,cv.obs_upper_error],color="red",marker="*",markersize=14,linestyle="none",capsize=5,label="PANORAMIC")
    plot_axis.set_xlabel(r"$M_{\rm UV}\ {\rm limit}$")
    plot_axis.set_xlim(-19.3,-20.7)
    plot_axis.set_ylim(bottom=0)
    plot_axis.grid(alpha=0.3)
    plot_axis.legend(fontsize=10)

ax[0].set_ylabel(r"$\sigma_{\rm CV}$")
ax[0].set_title("Central subhalos")
ax[1].set_title("All subhalos")

plt.tight_layout()
plt.show()

In [ ]:
#measured to linear cv ratio
fig,ax=plt.subplots(1,2,figsize=(14,5),facecolor="white",sharey=True)

for model_results,label,color,marker,linestyle in central_CV_models:
    MUV_limit=np.array([result["MUV_limit"] for result in model_results])
    ratio=np.array([result["cv_ratio"] for result in model_results])
    predicted=np.array([result["sigma_cv_predicted"] for result in model_results])
    error_low=np.array([result["sigma_cv_err_low"] for result in model_results])/predicted
    error_high=np.array([result["sigma_cv_err_high"] for result in model_results])/predicted
    ax[0].errorbar(MUV_limit,ratio,yerr=[error_low,error_high],color=color,marker=marker,linestyle=linestyle,capsize=4,label=label)

for model_results,label,color,marker,linestyle in all_CV_models:
    MUV_limit=np.array([result["MUV_limit"] for result in model_results])
    ratio=np.array([result["cv_ratio"] for result in model_results])
    predicted=np.array([result["sigma_cv_predicted"] for result in model_results])
    error_low=np.array([result["sigma_cv_err_low"] for result in model_results])/predicted
    error_high=np.array([result["sigma_cv_err_high"] for result in model_results])/predicted
    ax[1].errorbar(MUV_limit,ratio,yerr=[error_low,error_high],color=color,marker=marker,linestyle=linestyle,capsize=4,label=label)

for plot_axis in ax:
    plot_axis.axhline(1,color="black",linestyle="--")
    plot_axis.set_xlabel(r"$M_{\rm UV}\ {\rm limit}$")
    plot_axis.set_xlim(-19.3,-20.7)
    plot_axis.grid(alpha=0.3)
    plot_axis.legend(fontsize=10)

ax[0].set_ylabel(r"$\sigma_{\rm CV}^{\rm measured}/(b_{\rm eff}\sigma_m)$")
ax[0].set_title("Central subhalos")
ax[1].set_title("All subhalos")

plt.tight_layout()
plt.show()

In [ ]:
for MUV_limit in cv.MUV_limits:
    print(MUV_limit,np.count_nonzero((~lcdm_in_mass_range_all)&(lcdm_MUV_inside<MUV_limit)))